<!--
README.md
Shapefile Area Calculator (Automatic UTM Projection)
-->

<h1 align="center">🗺️ Shapefile Area Calculator (Automatic UTM Projection)</h1>

<p align="center">
  Upload a zipped shapefile and automatically compute its total area in km² using the appropriate UTM projection.
</p>

<hr/>

<h2>📌 Overview</h2>

<p>
This script allows users to:
</p>

<ul>
  <li>📂 Upload a <code>.zip</code> file containing a shapefile</li>
  <li>🔍 Automatically detect the correct <code>.shp</code> (even inside subfolders)</li>
  <li>🧭 Identify the appropriate UTM zone based on geometry centroid</li>
  <li>🔄 Reproject to a metric CRS</li>
  <li>📐 Calculate total area in square kilometers (<code>km²</code>)</li>
</ul>

<p>
Designed for use in <b>Google Colab</b>, but adaptable to local Python environments.
</p>

<hr/>

<h2>📂 Expected ZIP Content</h2>

<p>The uploaded <code>.zip</code> must contain at least:</p>

<pre><code>.shp
.shx
.dbf
.prj
</code></pre>

<p>
The script validates the shapefile integrity before processing.
</p>

<hr/>

<h2>⚙️ Workflow</h2>

<h3>1️⃣ Install Required Libraries</h3>

<pre><code>geopandas
pyogrio
pyproj
</code></pre>

<h3>2️⃣ Upload Shapefile ZIP</h3>

<ul>
  <li>User uploads a <code>.zip</code> via Colab interface</li>
  <li>The script extracts contents into a clean working directory</li>
  <li>Automatically finds the largest <code>.shp</code> file if multiple exist</li>
</ul>

<h3>3️⃣ Integrity Check</h3>

<p>
Ensures required shapefile components exist:
</p>

<pre><code>.shp
.shx
.dbf
.prj
</code></pre>

<p>
If missing, the script stops and informs the user.
</p>

<h3>4️⃣ Automatic UTM Detection</h3>

<p>
The script:
</p>

<ul>
  <li>Calculates the geometry centroid</li>
  <li>Determines correct UTM zone from longitude</li>
  <li>Detects hemisphere automatically</li>
  <li>Reprojects data into appropriate metric CRS</li>
</ul>

<p>
This guarantees accurate area calculation.
</p>

<h3>5️⃣ Area Calculation</h3>

<ul>
  <li>Geometries are unified (avoids overlap duplication)</li>
  <li>Area is computed in square meters</li>
  <li>Converted to <b>km²</b></li>
</ul>

<hr/>

<h2>📐 Output</h2>

<p>The script prints:</p>

<ul>
  <li>Original CRS</li>
  <li>Reprojected UTM CRS</li>
  <li>Total unified area in <b>km²</b></li>
</ul>

<hr/>

<h2>✨ Key Features</h2>

<ul>
  <li>✅ Fully automatic CRS handling</li>
  <li>✅ Works even if shapefile is inside subfolders</li>
  <li>✅ Automatic UTM zone detection</li>
  <li>✅ Hemisphere detection (north/south)</li>
  <li>✅ Safe geometry union before area computation</li>
  <li>✅ Handles missing <code>.shx</code> using fallback environment flag</li>
</ul>

<hr/>

<h2>🧠 Technical Notes</h2>

<ul>
  <li>UTM zone is computed as:<br/>
  <code>zone = int((longitude + 180) / 6) + 1</code></li>
  <li>Hemisphere determined from centroid latitude</li>
  <li>Area divided by <code>1e6</code> to convert m² → km²</li>
  <li><code>SHAPE_RESTORE_SHX=YES</code> allows reconstruction of missing index files</li>
</ul>

<hr/>

<h2>⚠️ Important Considerations</h2>

<ul>
  <li>This method assumes geometries fall within a single UTM zone.</li>
  <li>For very large regions spanning multiple zones, consider alternative projections.</li>
  <li>Geometries are unified before area calculation to prevent double counting.</li>
</ul>

<hr/>

<h2>🚀 Usage</h2>

<ol>
  <li>Run the notebook cell.</li>
  <li>Upload a valid shapefile ZIP.</li>
  <li>Wait for CRS detection and reprojection.</li>
  <li>Read total area printed in km².</li>
</ol>

<hr/>

<h2>🧪 Application Context</h2>

<p>
This workflow is suitable for:
</p>

<ul>
  <li>Coastal zone area estimation</li>
  <li>Intertidal zone mapping</li>
  <li>Remote sensing post-processing</li>
  <li>Morphodynamic studies</li>
</ul>

<p>
Developed for spatial analysis pipelines under coastal monitoring and topobathymetric research projects.
</p>

<hr/>

<h2>📄 License</h2>

<p>
CC BY 4.0
</p>

<hr/>

<p align="center">
  <sub>Automated, projection-safe area calculation for reproducible spatial analysis 🗺️📐</sub>
</p>

In [ ]:
# ============================================
# 1. Instalar bibliotecas (executar uma vez)
# ============================================
!pip install -q geopandas pyogrio

import geopandas as gpd
import os, glob, re
from google.colab import files
import pyproj
import shutil
import zipfile

# Fallback: tenta reconstruir .shx se necessário
os.environ["SHAPE_RESTORE_SHX"] = "YES"

# ============================================
# 2. Upload do ZIP
# ============================================
uploaded = files.upload()
zip_names = [k for k in uploaded.keys() if k.lower().endswith(".zip")]
if not zip_names:
    raise ValueError("Envie um arquivo .zip contendo o shapefile (.shp, .shx, .dbf, .prj).")

zip_path = zip_names[0]
print("ZIP recebido:", zip_path)

# ============================================
# 2.1 Extrair para pasta limpa
# ============================================
workdir = "work_shp"
if os.path.exists(workdir):
    shutil.rmtree(workdir)
os.makedirs(workdir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(workdir)

# ============================================
# 2.2 Encontrar o .shp (mesmo em subpastas)
# ============================================
shp_files = glob.glob(os.path.join(workdir, "**", "*.shp"), recursive=True)
if not shp_files:
    raise FileNotFoundError("Não encontrei nenhum .shp dentro do .zip. Verifique o conteúdo do arquivo.")

# Escolhe o maior .shp como padrão
shp_path = max(shp_files, key=lambda f: os.path.getsize(f))
base = os.path.splitext(shp_path)[0]
print("SHP encontrado:", shp_path)

# ============================================
# 2.3 Checar arquivos essenciais do shapefile
# ============================================
required_exts = [".shp", ".shx", ".dbf"]
missing = [ext for ext in required_exts if not os.path.exists(base + ext)]
if missing:
    print("Arquivos faltando:", missing)
    print("Arquivos encontrados no zip:", sorted(glob.glob(os.path.join(workdir, '**', '*.*'), recursive=True))[:30])
    raise FileNotFoundError("O shapefile está incompleto (faltando .shx e/ou .dbf). Recrie o .zip com todos os componentes.")

# ============================================
# 3. Ler shapefile
# ============================================
gdf = gpd.read_file(shp_path)
print("\nCRS original:", gdf.crs)

if gdf.empty:
    raise ValueError("O shapefile foi lido, mas está vazio (sem geometrias).")
    
# ============================================
# 4. Reprojetar automaticamente para UTM adequada
# ============================================
centroid = gdf.geometry.unary_union.centroid
lon, lat = centroid.x, centroid.y

utm_zone = int((lon + 180) / 6) + 1
utm_crs = pyproj.CRS.from_dict({'proj': 'utm', 'zone': utm_zone, 'south': lat < 0})

gdf_utm = gdf.to_crs(utm_crs)
print("CRS reprojetado:", gdf_utm.crs)

# ============================================
# 5. Calcular área em km²
# ============================================
# Unificar geometrias (evita sobreposição duplicada)
geom_unified = gdf_utm.unary_union

area_km2 = geom_unified.area / 1e6
print(f"\nÁrea total: {area_km2:.6f} km²")